In [ ]:
# ==========================================================
# STEP 1 : Import Required Libraries
#
# Purpose:
# Import LangGraph, LangChain, PromptTemplate and
# AWS Bedrock Chat Model.
#
# StateGraph  -> Builds the workflow
# START, END -> Graph entry and exit
# TypedDict  -> Defines shared state
# ==========================================================

from typing import TypedDict

from langgraph.graph import StateGraph, START, END

from langchain_core.prompts import ChatPromptTemplate

from langchain_aws import ChatBedrockConverse

In [ ]:
# ==========================================================
# STEP 2 : Create AWS Bedrock LLM
#
# Purpose:
# Create the Claude model.
#
# This model performs:
# 1. Tool Selection
# 2. Reasoning
# 3. Final Answer Generation
#
# Prerequisite:
# AWS Credentials should already be configured.
# ==========================================================

llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region_name="us-east-1"
)

In [ ]:
# ==========================================================
# STEP 3 : Define External Tools
#
# Purpose:
# In production these would be:
#
# RAG Tool
# --------
# Qdrant / OpenSearch
#
# HR API
# ------
# FastAPI / Microservice
#
# For learning we return dummy responses.
# ==========================================================

def rag_tool(question: str):

    print("\nUsing RAG Tool...\n")

    return """
HR Leave Policy

• Employees receive 20 annual leave days.
• Maximum 5 unused leaves can be carried forward.
• Leave requests require manager approval.
"""


def hr_api(question: str):

    print("\nCalling HR API...\n")

    return """
Employee : Suraj

Remaining Leave : 12 Days
"""

In [ ]:
# ==========================================================
# STEP 4 : Define LangGraph State
#
# Purpose:
# State stores information shared between nodes.
#
# question -> User Question
# tool     -> Selected Tool
# context  -> Tool Output
# answer   -> Final LLM Response
# ==========================================================

class AgentState(TypedDict):

    question: str

    tool: str

    context: str

    answer: str

In [ ]:
# ==========================================================
# STEP 5 : Create Prompt for Tool Selection
#
# Purpose:
# Claude decides which tool should be used.
#
# It should return ONLY
#
# rag
#
# api
#
# none
# ==========================================================

decision_prompt = ChatPromptTemplate.from_messages(

[
(
"system",
"""
You are an Enterprise HR Assistant.

Decide which tool should answer the question.

Available tools

1. rag
Use for:
- HR Policy
- Leave Policy
- Payroll Policy
- Company Rules

2. api
Use for:
- Employee Leave Balance
- Salary
- Employee Details

3. none
Use for:
- Greetings
- General Conversation

Reply ONLY with

rag

or

api

or

none
"""
),

("human","{question}")

]
)

decision_chain = decision_prompt | llm

In [ ]:
# ==========================================================
# STEP 6 : LangGraph Node
#
# Purpose:
# Ask Claude which tool should be used.
#
# Claude performs reasoning.
# ==========================================================

def decide_tool(state: AgentState):

    response = decision_chain.invoke(

        {
            "question": state["question"]
        }

    )

    decision = response.content.lower()

    print("\nClaude Decision :", decision)

    if "rag" in decision:

        tool = "rag"

    elif "api" in decision:

        tool = "api"

    else:

        tool = "none"

    return {

        "tool": tool

    }




In [ ]:

# ==========================================================
# STEP 7 : Execute Selected Tool
#
# Purpose:
# Execute whichever tool Claude selected.
#
# Production
#
# rag
#  -> Retriever(Qdrant)
#
# api
#  -> REST API
#
# none
#  -> No Tool
# ==========================================================

def execute_tool(state: AgentState):

    tool = state["tool"]

    question = state["question"]

    if tool == "rag":

        context = rag_tool(question)

    elif tool == "api":

        context = hr_api(question)

    else:

        context = "Hello! How can I help you today?"

    return {

        "context": context

    }

In [ ]:
# ==========================================================
# STEP 8 : Final Prompt
#
# Purpose:
# Give Claude
#
# Question
#
# +
#
# Tool Output(Context)
#
# Claude generates the final response.
# ==========================================================

answer_prompt = ChatPromptTemplate.from_messages(

[
(
"system",
"""
You are a helpful HR Assistant.

Answer naturally using the provided context.

If the context already contains the answer,
summarize it professionally.
"""
),

(
"human",
"""
Question

{question}


Context

{context}
"""
)

]

)

answer_chain = answer_prompt | llm

In [ ]:
# ==========================================================
# STEP 9 : Generate Final Answer
#
# Purpose:
# Claude receives:
#
# Question
#
# +
#
# Context
#
# and produces the final response.
# ==========================================================

def generate_answer(state: AgentState):

    response = answer_chain.invoke(

        {
            "question": state["question"],
            "context": state["context"]
        }

    )

    return {

        "answer": response.content

    }


In [ ]:
# ==========================================================
# STEP 10 : Build LangGraph Workflow
#
# Purpose:
# Create the workflow.
#
# START
#
# ↓
#
# Decide Tool
#
# ↓
#
# Execute Tool
#
# ↓
#
# Generate Answer
#
# ↓
#
# END
# ==========================================================

builder = StateGraph(AgentState)

builder.add_node("decide_tool", decide_tool)

builder.add_node("execute_tool", execute_tool)

builder.add_node("generate_answer", generate_answer)

builder.add_edge(START, "decide_tool")

builder.add_edge("decide_tool", "execute_tool")

builder.add_edge("execute_tool", "generate_answer")

builder.add_edge("generate_answer", END)

graph = builder.compile()

In [ ]:
# ==========================================================
# STEP 11 : Execute Workflow
#
# Purpose:
# Start the graph with the user's question.
# ==========================================================

result = graph.invoke(

{
    "question": "How many leave days do I have?",
    "tool": "",
    "context": "",
    "answer": ""
}

)


In [ ]:
# ==========================================================
# STEP 12 : Display Result
#
# Purpose:
# Print the final answer generated by Claude.
# ==========================================================

print("\n===============================")
print("FINAL ANSWER")
print("===============================\n")

print(result["answer"])


---

# Flow Diagram

```text
                User Question
                      │
                      ▼
              Claude (Reasoning)
                      │
         ┌────────────┼────────────┐
         ▼            ▼            ▼
       RAG          HR API       None
         │            │            │
         └────────────┼────────────┘
                      ▼
              Retrieved Context
                      │
                      ▼
          Claude (Generate Answer)
                      │
                      ▼
                 Final Response
```

## Interview Explanation

This example demonstrates the core idea of **Agentic RAG**:

1. **Claude reasons** about the user's request and selects the appropriate tool.
2. The selected tool returns the required context.
3. The context is passed back to **Claude**, which generates the final natural-language response.

Unlike **Traditional RAG**, retrieval is **optional**. The LLM dynamically decides whether to retrieve company policies, call an HR API, or respond directly.

> **Production Upgrade:** In an enterprise implementation, the `execute_tool()` node would typically be replaced with LangGraph's native tool-calling mechanism using `@tool`, `bind_tools()`, `ToolNode`, and `tools_condition()`. The RAG tool would query **Qdrant/OpenSearch**, and the HR API would call a real backend service rather than returning mocked data. This simplified version is ideal for understanding the architecture and for coding interviews.